# CDD-11-30: A0-L/A2-L 20-epoch controls

This notebook trains the 20-epoch controls sequentially. A0-L is the SIDD32 NAFNet baseline; A2-L adds the bottleneck adapter with skip gates. Both match A3-L's seed, data split, crop, optimizer, cosine schedule, and full-frame validation, with all auxiliary losses set to zero. Each run evaluates only its best-PSNR checkpoint. Expected runtime is approximately 45-60 minutes on two T4 GPUs.

Import this notebook from GitHub and attach only the existing `cdd-11-30` and `nafnetmodel` Kaggle inputs. The held-out test split is not evaluated.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/HoangKhanhTung0111/CoT-restoration.git"
REPO_DIR = Path("/kaggle/working/CoT-restoration")
if REPO_DIR.is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Commit:", COMMIT)
print("CWD:", Path.cwd())

In [ ]:
CDD11_ROOT = Path("/kaggle/input/datasets/mintesnotfikir/cdd-11-30")
PRETRAINED_ROOT = Path("/kaggle/input/datasets/hoangkhanhtung/nafnetmodel")
EXPERIMENTS_ROOT = Path("/kaggle/working/experiments_controls_long")
CONFIG = Path("configs/calibration_controls_long.json")
RUN_NAMES = [
    "a0l_nafnet_baseline_sidd32_seed42_20ep",
    "a2l_bottleneck_skip_sidd32_seed42_20ep",
]
assert CDD11_ROOT.is_dir(), f"Missing CDD-11 input: {CDD11_ROOT}"
assert PRETRAINED_ROOT.is_dir(), f"Missing pretrained input: {PRETRAINED_ROOT}"
assert CONFIG.is_file(), f"Missing config: {CONFIG}"
gpu_names = subprocess.check_output([
    "nvidia-smi", "--query-gpu=name", "--format=csv,noheader"
], text=True).strip().splitlines()
assert len(gpu_names) == 2 and all("T4" in name for name in gpu_names), (
    f"Select the Kaggle 2xT4 accelerator; found: {gpu_names}"
)
print("GPUs:", gpu_names)
print("CDD-11:", CDD11_ROOT)
print("Pretrained files:", sorted(path.name for path in PRETRAINED_ROOT.glob("*.pth")))

In [ ]:
# Recheck pairs, scene isolation, and exact pretrained compatibility.
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.audit_kaggle",
    "--data-root", str(CDD11_ROOT),
    "--pretrained-root", str(PRETRAINED_ROOT),
    "--output", "/kaggle/working/cot_nafnet_audit/audit_controls_long.json",
], check=True)

In [ ]:
# Change this to True only after the audit cell succeeds.
RUN_CONTROLS = False

In [ ]:
# Run A0-L first, then A2-L. Completed runs are skipped safely on rerun.
for run_name in RUN_NAMES:
    runner_command = [
        "python", "-m", "hybrid_cot_nafnet.run_ablation",
        "--config", str(CONFIG),
        "--data-root", str(CDD11_ROOT),
        "--experiments-root", str(EXPERIMENTS_ROOT),
        "--nproc-per-node", "2",
        "--runs", run_name,
    ]
    if RUN_CONTROLS:
        subprocess.run(runner_command, check=True)
    else:
        subprocess.run([*runner_command, "--dry-run"], check=True)
if not RUN_CONTROLS:
    print("Dry runs complete. Set RUN_CONTROLS = True and rerun from this cell.")

In [ ]:
from IPython.display import display
import pandas as pd

for run_name in RUN_NAMES:
    run_dir = EXPERIMENTS_ROOT / run_name
    for label, path in {
        "training": run_dir / "run_summary.json",
        "best restoration": run_dir / "evaluation" / "summary.json",
    }.items():
        if path.is_file():
            print(f"\n{run_name} - {label}:")
            display(json.loads(path.read_text()))
    train_log = run_dir / "train_log.csv"
    if train_log.is_file():
        display(pd.read_csv(train_log))

In [ ]:
import zipfile
from IPython.display import FileLink, FileLinks

summary_csv = EXPERIMENTS_ROOT / "ablation_summary.csv"
completed_runs = [
    run_name for run_name in RUN_NAMES
    if (EXPERIMENTS_ROOT / run_name / "evaluation" / "summary.json").is_file()
    and (EXPERIMENTS_ROOT / run_name / "run_summary.json").is_file()
    and json.loads((EXPERIMENTS_ROOT / run_name / "run_summary.json").read_text()).get("status") == "completed"
    and json.loads((EXPERIMENTS_ROOT / run_name / "run_summary.json").read_text()).get("completed_epochs") == 20
]
if summary_csv.is_file() and completed_runs == RUN_NAMES:
    display(pd.read_csv(summary_csv))
    lightweight = Path("/kaggle/working/controls_long_lightweight_results.zip")
    with zipfile.ZipFile(lightweight, "w", compression=zipfile.ZIP_DEFLATED) as output_zip:
        for path in sorted(EXPERIMENTS_ROOT.rglob("*")):
            if path.is_file() and path.suffix.lower() not in {".pt", ".png", ".jpg", ".jpeg"}:
                output_zip.write(path, path.relative_to(EXPERIMENTS_ROOT))

    comparisons = Path("/kaggle/working/controls_long_comparisons.zip")
    with zipfile.ZipFile(comparisons, "w", compression=zipfile.ZIP_DEFLATED) as output_zip:
        for run_name in RUN_NAMES:
            comparison_root = EXPERIMENTS_ROOT / run_name / "evaluation" / "comparisons"
            for path in sorted(comparison_root.glob("*.png")):
                output_zip.write(path, Path(run_name) / path.name)

    print("Download both ZIP files for the A0-L/A2-L comparison:")
    display(FileLink(str(lightweight)))
    display(FileLink(str(comparisons)))
else:
    print(f"Both 20-epoch controls and evaluations are required before export; completed: {completed_runs}")

if EXPERIMENTS_ROOT.is_dir():
    display(FileLinks(str(EXPERIMENTS_ROOT)))